# 02 — Chunking Experiment

**Talk role:** Establishes the 32-frame sampling problem and quantifies when scene chunking helps vs. hurts.

v5-omni samples 32 evenly-spaced frames per video, regardless of length. So:
- A 10s clip → 1 frame per 0.3s (essentially captures everything)
- A 60s clip → 1 frame per ~1.9s (some shots may be skipped)
- A 180s clip → 1 frame per ~5.6s (entire fast cuts lost)

**Hypothesis:** scene-based chunking should outperform whole-clip embedding on long clips, and the crossover should land somewhere around 30-45 seconds.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve() / 'scripts'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from embed_jina import JinaClient, EmbedConfig
from index_elastic import es_client, knn_search, index_name
from evaluate import load_eval_set, run_config

sns.set_style('whitegrid')
jina = JinaClient()
es = es_client()
queries, relevance = load_eval_set(Path('../data/queries.json'), Path('../data/relevance.json'))

## Run the same queries against three chunking strategies

All three use baseline 1024-d float32, so the *only* variable is chunking strategy.

In [ ]:
def search_fn_for(strategy: str):
    name = index_name(strategy, 1024, 'float')
    cfg = EmbedConfig(dimensions=1024)
    def fn(text):
        qvec = jina.embed([text], task='retrieval.query', config=cfg)[0]
        return knn_search(es, name, qvec, k=10)
    return fn

reports = []
for strategy in ['whole', 'scene', 'fixed30']:
    name = index_name(strategy, 1024, 'float')
    if not es.indices.exists(index=name):
        print(f'⚠ Skipping {strategy} — index missing')
        continue
    n_docs = es.count(index=name)['count']
    reports.append(run_config(
        label=strategy, dims=1024, quantization='float', num_docs=n_docs,
        queries=queries, relevance=relevance, search_fn=search_fn_for(strategy),
    ))

df = pd.DataFrame([r.summary() for r in reports])
df[['label', 'n_queries', 'recall@5', 'recall@10', 'mrr', 'latency_p50_ms']]

## Break down by parent-clip duration

The interesting finding lives here: scene chunking should win on long clips, lose on short ones.

In [ ]:
# This requires you to track parent-clip duration in your eval set.
# Add a 'duration_bucket' field to each query indicating the duration of
# its ground-truth target clip(s).
#
# Buckets: 'short' (<30s), 'medium' (30-90s), 'long' (>90s)
#
# Then group reports by bucket and recompute recall.

import json
with open('../data/queries.json') as f:
    bucket_map = {q['id']: q.get('duration_bucket', 'unknown')
                  for q in json.load(f)['queries']}

rows = []
for r in reports:
    for qr in r.results:
        rows.append({
            'strategy': r.label,
            'bucket':   bucket_map.get(qr.query_id, 'unknown'),
            'recall5':  qr.recall_at(5),
            'recall10': qr.recall_at(10),
        })

by_bucket = pd.DataFrame(rows).groupby(['strategy', 'bucket']).mean(numeric_only=True)
by_bucket

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
pivoted = pd.DataFrame(rows).pivot_table(
    index='bucket', columns='strategy', values='recall10', aggfunc='mean'
)
pivoted.plot(kind='bar', ax=ax, color=['#0d6efd', '#198754', '#fd7e14'])
ax.set_ylabel('Recall@10')
ax.set_title('Chunking strategy × clip duration')
ax.set_xlabel('Source clip duration')
plt.xticks(rotation=0); plt.tight_layout(); plt.show()